# Caption here


In [1]:
import geopandas as gpd 
from pathlib import Path
import scanpy as sc 

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Importing the embeddings for both modalities

In [3]:
# Variables
sample = "CRC_P2"
bin_size = "002um"
cellseg_method = "cell_seg_stardist" # in case of multiple cell segmentation methods, this determines which method we work with

# Paths/Directories
basedir = Path("../../data")
hist_dir = basedir / "combined/01_preprocessing/unimodel/metadata"                               # Note: these directories refers to the combined folder. 
trans_dir = basedir / "combined/01_preprocessing" / cellseg_method / bin_size / "custom_binning" # If you want to access a single slide folder, please define an extra path. 
output_dir = basedir / sample / "03_integration/autoencoders" / bin_size
output_dir.mkdir(parents=True, exist_ok=True)

# Files and filenames
hist_adata = sc.read_h5ad(hist_dir / "filtered_WSI_patch_embeddings.h5ad")
trans_adata = sc.read_h5ad(trans_dir / "patch_assigned_count_area_filtered_adata.h5ad")
trans_coord = gpd.read_parquet(trans_dir / "nuclei_geometry.parquet")

## Filter adata for single sample
hist_adata = hist_adata[hist_adata.obs["Slide_ID"] == sample]
trans_adata = trans_adata[trans_adata.obs["Slide_ID"] == sample]
trans_coord = trans_coord[trans_coord["Slide_ID"] == sample]

hist_obs, hist_features  = hist_adata.shape
trans_obs, trans_features = trans_adata.shape
print("Observation (patch) count histology data: ", hist_obs, " with feature vector length ", hist_features, ".")
print("Observation (nuclei/bin) count transcriptomics data: ", trans_obs, " with feature vector length ", trans_features, ".")

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Observation (patch) count histology data:  9572  with feature vector length  1536 .
Observation (nuclei/bin) count transcriptomics data:  144296  with feature vector length  18085 .


In [4]:
trans_coord

,id,geometry,Slide_ID,centroid,interior_centroid
59192,ID_9187,"POLYGON ((44622.832 3936.000, 44622.677 3938.1...",CRC_P2,POINT (44611.827 3936.127),POINT (44611.582 3937.062)
59193,ID_9205,"POLYGON ((44543.980 3940.000, 44542.817 3941.3...",CRC_P2,POINT (44535.136 3940.559),POINT (44535.492 3940.678)
59194,ID_9225,"POLYGON ((44565.494 3950.000, 44565.084 3952.2...",CRC_P2,POINT (44551.988 3951.590),POINT (44551.772 3951.102)
59195,ID_9268,"POLYGON ((44645.066 3934.000, 44645.586 3935.9...",CRC_P2,POINT (44638.654 3934.697),POINT (44638.366 3934.433)
59196,ID_9275,"POLYGON ((44585.070 3938.000, 44584.239 3938.8...",CRC_P2,POINT (44576.256 3938.473),POINT (44576.209 3938.422)
...,...,...,...,...,...
203483,ID_838110,"POLYGON ((64623.260 21968.000, 64623.476 21969...",CRC_P2,POINT (64614.459 21967.866),POINT (64614.011 21967.531)
203484,ID_838150,"POLYGON ((64476.381 21850.000, 64475.686 21852...",CRC_P2,POINT (64466.349 21850.097),POINT (64466.270 21849.299)
203485,ID_838165,"POLYGON ((64666.224 22300.000, 64664.501 22303...",CRC_P2,POINT (64654.302 22301.627),POINT (64654.176 22301.547)
203486,ID_838170,"POLYGON ((64550.979 21210.000, 64550.734 21210...",CRC_P2,POINT (64544.993 21207.860),POINT (64544.671 21207.321)


In [4]:
trans_adata.obs

,id,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,clusters,Slide_ID,Patch_ID
59192,ID_9187,335,5.817111,376.0,5.932245,24.202128,37.500000,64.095745,100.000000,5,CRC_P2,patch_1735.png
59193,ID_9205,438,6.084499,507.0,6.230481,23.471400,33.333333,53.057199,100.000000,5,CRC_P2,patch_1735.png
59194,ID_9225,867,6.766192,1139.0,7.038784,22.827041,31.606673,41.439860,67.778753,5,CRC_P2,patch_1735.png
59195,ID_9268,182,5.209486,204.0,5.323010,35.294118,59.803922,100.000000,100.000000,0,CRC_P2,patch_1836.png
59196,ID_9275,207,5.337538,240.0,5.484797,34.583333,55.416667,97.083333,100.000000,5,CRC_P2,patch_1735.png
...,...,...,...,...,...,...,...,...,...,...,...,...
203483,ID_838110,245,5.505332,281.0,5.641907,30.604982,48.398577,83.985765,100.000000,5,CRC_P2,patch_10906.png
203484,ID_838150,214,5.370638,245.0,5.505332,33.061224,53.469388,94.285714,100.000000,0,CRC_P2,patch_10804.png
203485,ID_838165,121,4.804021,126.0,4.844187,43.650794,83.333333,100.000000,100.000000,0,CRC_P2,patch_10907.png
203486,ID_838170,228,5.433722,255.0,5.545177,30.196078,49.803922,89.019608,100.000000,5,CRC_P2,patch_10801.png


In [5]:
hist_adata

View of AnnData object with n_obs × n_vars = 9572 × 1536
    obs: 'Slide_ID', 'Patch_ID', 'X', 'Y', 'leiden_0.2', 'leiden_0.5', 'leiden_1.0', 'leiden_2.0'
    uns: 'leiden_0.2', 'leiden_0.2_colors', 'leiden_0.5', 'leiden_0.5_colors', 'leiden_1.0', 'leiden_1.0_colors', 'leiden_2.0', 'leiden_2.0_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [13]:
trans_adata

AnnData object with n_obs × n_vars = 456881 × 18085
    obs: 'id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'clusters', 'Slide_ID', 'Patch_ID'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    obsm: 'X_pca'
    varm: 'PCs'

Questions:
1. Do we add the patch embedding to each nucleus in the patch but encode per nucleus, or do we summarise the nuclei per patch and then combine one patch embedding + one larger nuclei embedding?
2. do patch-specific graphs make sense?